# Fongbe ASR — KenLM sur le modèle v2

1. Entraîne un KenLM 4-grammes sur **toutes les transcriptions** (reliable + religious + uncertain)
2. Décode les 300 voix de validation avec `pyctcdecode` + LM
3. Compare le TSER officiel : **greedy vs KenLM**

Prérequis : runtime GPU recommandé (inférence rapide), CPU suffit sinon.

In [ ]:
# === 0. Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
os.makedirs('/content/work', exist_ok=True)

# deps système pour compiler kenlm (binaires + bindings python)
!apt-get install -y -q cmake build-essential libbz2-dev zlib1g-dev libboost-all-dev libeigen3-dev >/dev/null 2>&1
# libs
!pip install -q "transformers==5.14.1" soundfile pyctcdecode rapidfuzz "https://github.com/kpu/kenlm/archive/master.zip"
# binaires kenlm (lmplz / build_binary)
if not os.path.exists('/content/kenlm-src/build/bin/lmplz'):
    !git clone -q --depth 1 https://github.com/kpu/kenlm /tmp/kenlm-src 2>/dev/null || true
    import shutil as _sh, os as _os
    if _os.path.exists('/content/kenlm-src'): _sh.rmtree('/content/kenlm-src')
    if _os.path.exists('/tmp/kenlm-src'): _sh.move('/tmp/kenlm-src', '/content/kenlm-src')
    !mkdir -p /content/kenlm-src/build && cd /content/kenlm-src/build && cmake .. -DCMAKE_BUILD_TYPE=Release && make -j$(nproc)
    print('build/bin:', os.listdir('/content/kenlm-src/build/bin') if os.path.exists('/content/kenlm-src/build/bin') else 'ABSENT')
if os.path.lexists('/content/kenlm-bin'):
    os.remove('/content/kenlm-bin')
os.symlink('/content/kenlm-src/build/bin', '/content/kenlm-bin')
print('lmplz:', os.path.exists('/content/kenlm-bin/lmplz'))

In [ ]:
# === 1. Données : CSV + scripts depuis Drive, val depuis GitHub ===
import urllib.request, shutil
from pathlib import Path

DATA = Path('/content/fongbe'); DATA.mkdir(exist_ok=True)
SRC = Path('/content/drive/MyDrive/fongbe-challenge/data')
for f in ['Train_reliable.csv','Train_uncertain.csv','Train_religious.csv',
          'syllabify.py','normalize.py','score.py']:
    shutil.copy2(SRC / f, DATA / f)
    print('✓', f)
BASE = 'https://raw.githubusercontent.com/Appolinairee/fongbe-challenge/master'
for f in ['valid_reference.csv','valid_prediction.csv']:
    urllib.request.urlretrieve(f'{BASE}/val/{f}', DATA / f)
print('csv ok')

In [ ]:
# === 2. Corpus KenLM : toutes les transcriptions normalisées ===
import pandas as pd, sys
sys.path.insert(0, str(DATA))
from normalize import tonal_target

frames = [pd.read_csv(DATA / f) for f in
          ['Train_reliable.csv','Train_religious.csv','Train_uncertain.csv']]
all_df = pd.concat(frames, ignore_index=True)
print(len(all_df), 'transcriptions')

with open('/content/lm_corpus.txt', 'w', encoding='utf-8') as f:
    for t in all_df.transcription.fillna(''):
        s = tonal_target(t).strip()
        if s:
            f.write(s + '\n')
print('corpus écrit')

In [ ]:
# === 3. KenLM : build binaire + bindings python ===
import subprocess, os
os.makedirs('/content/lm', exist_ok=True)

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print('$', cmd[:140]); print('rc=', r.returncode)
    if r.stdout: print(r.stdout[-2500:])
    if r.stderr: print('STDERR:', r.stderr[-2500:])
    return r

# lm.arpa existe déjà (lmplz a tourné) ; sinon on le reconstruit
if not os.path.exists('/content/lm/lm.arpa'):
    with open('/content/lm_corpus.txt') as fin, open('/content/lm/lm.arpa','w') as fout:
        subprocess.run(['/content/kenlm-bin/lmplz','-o','4'], stdin=fin, stdout=fout, check=True)

# build_binary : PAS de flag -o, l'ordre est dans le header ARPA.
# 'trie' utilise le moins de RAM ( probing = plus rapide mais plus gourmand )
if not os.path.exists('/content/lm/lm.bin'):
    sh('/content/kenlm-bin/build_binary -T /content/lm -S 50% trie /content/lm/lm.arpa /content/lm/lm.bin')
if os.path.exists('/content/lm/lm.bin'):
    print('LM OK:', os.path.getsize('/content/lm/lm.bin')/1e6, 'Mo')

# bindings python kenlm (requis par pyctcdecode) — incompatibles py3.13
# (Cython pré-généré). On crée un venv Python 3.12 via uv et on y installe
# pyctcdecode + kenlm ; le décodage tournera en sous-processus.
if not os.path.exists('/content/venv312/bin/python'):
    sh('pip install -q uv')
    sh('uv venv --python 3.12 --seed /content/venv312')
    sh('uv pip install --python /content/venv312/bin/python pyctcdecode')
    sh('uv pip install --python /content/venv312/bin/python "kenlm @ https://github.com/kpu/kenlm/archive/master.zip" 2>&1 | tail -25')
sh('/content/venv312/bin/python -c "import kenlm, pyctcdecode"')


In [ ]:
# === 4. Audio de validation : les 300 wav du split v2 ===
import json, subprocess
val = pd.read_csv(DATA / 'valid_reference.csv')
rel = pd.read_csv(DATA / 'Train_reliable.csv').set_index('id')
val['audio_path'] = val.id.map(rel.audio_path)
print(len(val), 'fichiers val')

if not Path('/content/audio/train_reliable').exists():
    listing = json.loads(urllib.request.urlopen(
        'https://storage.googleapis.com/storage/v1/b/fongbe/o?maxResults=200&fields=items(name)').read())
    tars = sorted(i['name'] for i in listing['items']
                  if i['name'].startswith('audio_train_reliable'))
    for name in tars:
        tp = f'/content/{name}'
        if not Path(tp).exists():
            print('download', name, flush=True)
            subprocess.run(['wget','-q','-O',tp,f'https://storage.googleapis.com/fongbe/{name}'], check=True)
        subprocess.run(['tar','-xf',tp,'-C','/content'], check=True)
        os.remove(tp)
    Path('/content/audio').rename('/content/fongbe/audio') if Path('/content/audio').exists() and not Path('/content/fongbe/audio').exists() else None
print('audio ok')

In [ ]:
# === 5. Modèle v2 + décodage greedy vs KenLM ===
import torch, soundfile as sf, numpy as np
from transformers import Wav2Vec2BertProcessor, Wav2Vec2BertForCTC

# localise le modèle v2 sur Drive
MODEL_DIR = '/content/drive/MyDrive/fongbe/model_w2vbert_v2'
assert os.path.exists(MODEL_DIR + '/model.safetensors'), MODEL_DIR
print('modèle:', MODEL_DIR)
assert MODEL_DIR

device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = Wav2Vec2BertProcessor.from_pretrained(MODEL_DIR)
model = Wav2Vec2BertForCTC.from_pretrained(MODEL_DIR).to(device).eval()

# labels bruts -> pyctcdecode normalise ('|'->' ', [PAD]->blank, [UNK]->unk)
import json as _json
vocab = _json.loads(open(MODEL_DIR + '/vocab.json').read())
id2tok = {i: t for t, i in vocab.items()}
n = max(id2tok) + 1
labels = [id2tok.get(i, '') for i in range(n)]
_json.dump(labels, open('/content/labels.json','w'))

# unigrammes connus du LM (évite le warning "unigrams not provided" avec .bin)
uni = set()
for line in open('/content/lm_corpus.txt', encoding='utf-8'):
    for w in line.split():
        uni.add(w)
open('/content/unigrams.txt','w',encoding='utf-8').write('\n'.join(sorted(uni)))

decode_src = """import json, glob
import numpy as np
from pyctcdecode import build_ctcdecoder
labels = json.load(open('/content/labels.json'))
unigrams = set(open('/content/unigrams.txt', encoding='utf-8').read().split())
dec = build_ctcdecoder(labels, kenlm_model_path='/content/lm/lm.bin',
                       unigrams=unigrams, alpha=0.5, beta=1.5)
files = sorted(glob.glob('/content/lm_logits/*.npy'))
out = []
for i, f in enumerate(files):
    lg = np.load(f).astype(np.float32)
    out.append(dec.decode(lg))
    if (i + 1) % 25 == 0:
        print(i + 1, '/', len(files), flush=True)
json.dump(out, open('/content/lm_hyps.json', 'w'), ensure_ascii=False)
print('decoded', len(out))
"""
open('/content/decode_lm.py', 'w').write(decode_src)
print('labels/unigrams/script prêts')

In [ ]:
import json, subprocess
# === 6. Inférence + score des deux décodages ===
import sys as _s, subprocess as _sp
_sp.run([_s.executable, '-m', 'pip', 'install', '-q', 'rapidfuzz'], check=True)
_s.path.insert(0, str(DATA))
import score as official

AUDIO = Path('/content/fongbe/audio')
import os as _os
import shutil as _sh
for _f in ['/content/lm_hyps.json']:
    if os.path.exists(_f): os.remove(_f)
_sh.rmtree('/content/lm_logits', ignore_errors=True)
_os.makedirs('/content/lm_logits', exist_ok=True)
greedy_hyps, refs = [], []

@torch.no_grad()
def logits_for(path):
    # chunks de 20 s : évite le coût quadratique de l'attention sur longs fichiers
    x, sr = sf.read(str(path), dtype='float32')
    if x.ndim > 1: x = x.mean(axis=1)
    CH = 16000 * 20
    outs = []
    for st in range(0, len(x), CH):
        seg = x[st:st+CH]
        if len(seg) < 1600:
            seg = np.pad(seg, (0, 1600 - len(seg)))
        f = processor(seg, sampling_rate=16000, return_tensors='pt')
        outs.append(model(input_features=f.input_features.to(device),
                           attention_mask=f.attention_mask.to(device)).logits[0].float().cpu().numpy())
    return np.concatenate(outs, axis=0)

for k, row in enumerate(val.itertuples()):
    npy = f'/content/lm_logits/{k:03d}.npy'
    if not os.path.exists(npy):
        wav = AUDIO / row.audio_path.replace('audio/', '', 1)
        lg = logits_for(wav)
        np.save(npy, lg.astype(np.float16))
    try:
        lg = np.load(npy).astype(np.float32)
    except Exception:
        os.remove(npy)
        wav = AUDIO / row.audio_path.replace('audio/', '', 1)
        lg = logits_for(wav)
        np.save(npy, lg.astype(np.float16))
        lg = lg.astype(np.float32)
    greedy_hyps.append(processor.batch_decode(lg.argmax(-1)[None], skip_special_tokens=True)[0])
    refs.append(row.transcription)
    if (k+1) % 25 == 0: print(k+1, '/300', flush=True)

# décodage LM dans le venv py3.12 (bindings kenlm)
print('lancement decode_lm.py (venv312)...', flush=True)
subprocess.run(['/content/venv312/bin/python', '/content/decode_lm.py'], check=True)
lm_hyps = json.load(open('/content/lm_hyps.json'))
assert len(lm_hyps) == len(refs), (len(lm_hyps), len(refs))

pairs_g = list(zip(refs, greedy_hyps))
pairs_l = list(zip(refs, lm_hyps))
mg = official.measure(pairs_g); ml = official.measure(pairs_l)
print('GREEDY :', {k: round(v,4) for k,v in mg.items() if k in ('score','TSER','TER_conditional')})
print('KENLM  :', {k: round(v,4) for k,v in ml.items() if k in ('score','TSER','TER_conditional')})

In [ ]:
# === 7. Décodage LM (si besoin) + score + sauvegarde Drive ===
import json, subprocess, os
if not os.path.exists('/content/lm_hyps.json'):
    print('lancement decode_lm.py (venv312)...', flush=True)
    r = subprocess.run(['/content/venv312/bin/python', '/content/decode_lm.py'],
                       capture_output=True, text=True)
    print(r.stdout[-3000:])
    print('STDERR:', r.stderr[-4000:])
    assert r.returncode == 0, 'decode_lm.py a échoué'
lm_hyps = json.load(open('/content/lm_hyps.json'))
assert len(lm_hyps) == len(refs), (len(lm_hyps), len(refs))

pairs_g = list(zip(refs, greedy_hyps))
pairs_l = list(zip(refs, lm_hyps))
mg = official.measure(pairs_g); ml = official.measure(pairs_l)
print('GREEDY :', {k: round(v,4) for k,v in mg.items() if k in ('score','TSER','TER_conditional')})
print('KENLM  :', {k: round(v,4) for k,v in ml.items() if k in ('score','TSER','TER_conditional')})

res = pd.DataFrame({'id': val.id, 'ref': refs, 'greedy': greedy_hyps, 'kenlm': lm_hyps})
res.to_csv('/content/kenlm_val_results.csv', index=False)
DEST = '/content/drive/MyDrive/fongbe-kenlm'
os.makedirs(DEST, exist_ok=True)
for f in ['/content/lm/lm.bin','/content/lm/lm.arpa','/content/kenlm_val_results.csv']:
    subprocess.run(['cp', f, DEST + '/' + os.path.basename(f)], check=False)
print('sauvegardé dans Drive/fongbe-kenlm')
